# Chapter 5: Symbolic Derivation on Manifolds with `geomech`

This notebook demonstrates the complete `geomech` pipeline (Kotaru, 2026) implementing coordinate-free variational mechanics on $\mathbb{R}^3$, $S^2$, $SO(3)$, and $SE(3) \times S^2$:
1. **Spherical Pendulum on $S^2$**: `compute_eom`, `EOMTrace`, `check_eom`, and `RKMK4` energy conservation.
2. **Rigid Body on $SO(3)$**: `to_standard_form` ($M\dot{\Omega} + f + Gu = 0$).
3. **Quadrotor with Cable-Suspended Load on $SE(3) \times S^2$**: Automated derivation matching Chapter 4 Eqs. (4.16)–(4.21) + `linearize`.
4. **Symbolic $n$-Body Chain (`geomech.indexed`)**: Generic $i$-th body equation and instantiation at $n=3$.

In [ ]:
# Install geomech in Colab if needed
!pip install -q "geomech[sim]"

In [ ]:
import numpy as np
import geomech
from geomech import (
    S2, SO3, Cross, Dot, Matrix, Scalar, SystemVariables, Vector,
    EOMTrace, check_eom, compute_eom, getScalars, linearize, print_eom, to_standard_form
)
from geomech.codegen import StateLayout, collect_params, load_module, module_source
from geomech.simulation import simulate

print("geomech version:", geomech.__version__)

## 1. Spherical Pendulum on $S^2$ + `EOMTrace` + `RKMK4` Energy Drift

In [ ]:
half = Scalar("0.5", value=0.5, attr=["Constant"])
m, g, l = getScalars("m g l", attr=["Constant"])
e3 = Vector("e3", attr=["Constant"])
q, M = S2("q"), Vector("M")
omega, xi = q.get_tangent_vector(), q.get_variation_vector()

L = half * m * l * l * Dot(omega, omega) - m * g * l * Dot(q, e3)
dW = Dot(xi, M)
variables, inputs = SystemVariables(vectors=[q]), [M]

tr = EOMTrace()
eom = compute_eom(L, dW, variables, trace=tr)
err = check_eom(L, dW, variables, eom, inputs=inputs)
print(f"check_eom residual: {err:.2e}")
assert err < 1e-5
tr.summary(chars=72)
print_eom(eom)

In [ ]:
sf = to_standard_form(eom, variables, inputs)
layout = StateLayout(variables, inputs, collect_params([L, dW], variables, inputs))
plant = load_module(module_source(sf, layout, lagrangian=L), "spherical_pendulum")

p = {"e3": np.array([0.0, 0.0, 1.0]), "g": 9.81, "l": 1.0, "m": 1.0}
q0 = np.array([np.sin(1.0), 0.0, -np.cos(1.0)])
w0 = np.array([0.0, 1.5, 0.0])
w0 -= (w0 @ q0) * q0

traj = simulate(plant, layout.pack(q=q0, omega_q=w0), 5.0, p)
E = traj.energy()
drift = np.max(np.abs(E - E[0])) / abs(E[0])
print(f"RKMK4 relative energy drift over 5s: {drift:.2e}")
print(f"Max ||q|| - 1: {np.max(np.abs(np.linalg.norm(traj.state[:, :3], axis=1) - 1.0)):.2e}")

## 2. Quadrotor with Cable-Suspended Load on $SE(3) \times S^2$ & Linearization

In [ ]:
mQ, mL = getScalars("m_Q m_L", attr=["Constant"])
J = Matrix("J", attr=["Constant", "SymmetricMatrix"])
xL, q, R = Vector("x_L"), S2("q"), SO3("R")
xi = q.get_variation_vector()
Om, eta = R.get_tangent_vector(), R.get_variation_vector()
f, M_tau = Scalar("f"), Vector("M")

xQ = xL - q * l
vQ, vL = xQ.t_diff(), xL.t_diff()
L_ql = (
    half * mQ * Dot(vQ, vQ)
    + half * mL * Dot(vL, vL)
    + half * Dot(Om, J * Om)
    - mQ * g * Dot(e3, xQ)
    - mL * g * Dot(e3, xL)
)
dW_ql = Dot(xL.delta() - Cross(xi, q) * l, (R * e3) * f) + Dot(eta, M_tau)
vars_ql = SystemVariables(vectors=[xL, q], matrices=[R])

eom_ql = compute_eom(L_ql, dW_ql, vars_ql)
err_ql = check_eom(L_ql, dW_ql, vars_ql, eom_ql, inputs=[f, M_tau])
print(f"Quadrotor-Load check_eom residual: {err_ql:.2e}")
assert err_ql < 1e-5

lin_ql = linearize(eom_ql, vars_ql, [f, M_tau])
print("Variation-based linearization derived successfully for SE(3) x S^2.")

## 3. Symbolic $n$-Link Spherical Chain (`geomech.indexed`)

In [ ]:
from geomech.indexed import Family, IndexedVariables, Sum, compute_indexed_eom, indices, n

i, j = indices("i j")
q_fam = Family(S2, "q")
m_fam = Family(Scalar, "m", attr=["Constant"])
l_fam = Family(Scalar, "l", attr=["Constant"])
M_fam = Family(Vector, "M")

def pos_k(k):
    return Sum(j, 1, k, q_fam[j] * l_fam[j])

v_i = pos_k(i).t_diff()
L_n = Sum(i, 1, n, half * m_fam[i] * v_i.dot(v_i)) - Sum(i, 1, n, m_fam[i] * g * pos_k(i).dot(e3))
dW_n = Sum(i, 1, n, Dot(q_fam[i].get_variation_vector(), M_fam[i]))

eom_n = compute_indexed_eom(L_n, dW_n, IndexedVariables(families=[q_fam]))
print("Generic i-th body LaTeX:")
print(eom_n["q"].to_latex())

eom_3 = eom_n.instantiate(3)
print("Instantiated 3-link equations:", list(eom_3.keys()))